# 01 — Data exploration

Which `nba_api` endpoints give us clean, league-wide, time-correct player-season data? Everything here is tested on **one season** before any bulk download.

In [1]:
import pandas as pd
import numpy as np

## Step 1B — League-wide player-season data: `LeagueDashPlayerStats`

One call returns **every player for one season**, so 15 seasons × a few measure types is about 50–100 calls total.

Design choice: request `Totals`, not `Per36`. Totals add up across seasons and stints, so we can derive per-36, per-100, rolling multi-season rates and shrinkage estimates ourselves from one consistent source. Advanced rates (USG%, AST%, …) need team and opponent possession data we don't have, so we take those as given.

⚠️ With `Advanced`, `MIN` is **per game** even when `Totals` is requested. We take minutes from `Base` only.

In [2]:
import time
from nba_api.stats.endpoints import leaguedashplayerstats as ldps

SEASON = "2018-19"

def fetch_league_players(season, measure, per_mode="Totals"):
    """One call = every player in the league for one season (consolidated across teams)."""
    df = ldps.LeagueDashPlayerStats(
        season=season,
        season_type_all_star="Regular Season",
        measure_type_detailed_defense=measure,
        per_mode_detailed=per_mode,
        timeout=60,
    ).get_data_frames()[0]
    time.sleep(1.0)  # be polite to stats.nba.com; it rate-limits aggressively
    return df

base = fetch_league_players(SEASON, "Base")
adv = fetch_league_players(SEASON, "Advanced")

print(base.shape, adv.shape)
print([c for c in base.columns if not c.endswith("_RANK")])

(530, 67) (530, 79)
['PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS', 'NBA_FANTASY_PTS', 'DD2', 'TD3', 'WNBA_FANTASY_PTS', 'TEAM_COUNT']


In [3]:
adv_cols = ["PLAYER_ID", "POSS", "PACE", "USG_PCT", "TS_PCT", "EFG_PCT",
            "AST_PCT", "AST_RATIO", "TM_TOV_PCT", "OREB_PCT", "DREB_PCT", "REB_PCT",
            "OFF_RATING", "DEF_RATING", "NET_RATING", "PIE"]

df = base.merge(adv[adv_cols], on="PLAYER_ID", how="outer",
                validate="one_to_one", indicator=True)

print("shape:", df.shape)
print("join:", df["_merge"].value_counts().to_dict())
print("teams per player:", df["TEAM_COUNT"].value_counts().sort_index().to_dict())
print("players with >= N minutes:",
      {t: int((df["MIN"] >= t).sum()) for t in [0, 100, 250, 500, 1000, 1500]})

shape: (530, 83)
join: {'both': 530, 'left_only': 0, 'right_only': 0}
teams per player: {1: 444, 2: 80, 3: 6}
players with >= N minutes: {0: 530, 100: 461, 250: 413, 500: 361, 1000: 276, 1500: 183}


In [4]:
cols = ["PLAYER_NAME", "GP", "MIN", "FGA", "FG3A", "FG3_PCT", "TS_PCT", "USG_PCT"]
display(df.sort_values("TS_PCT", ascending=False)[cols].head(5))
display(df[df["PLAYER_NAME"] == "Shai Gilgeous-Alexander"][cols + ["AST_PCT", "POSS"]])

,PLAYER_NAME,GP,MIN,FGA,FG3A,FG3_PCT,TS_PCT,USG_PCT
306,Jordan Sibert,1,4.233333,1,1,1.0,1.500,0.083
328,Zhou Qi,1,0.955000,1,0,0.0,1.000,0.333
456,Trevon Duval,3,5.603333,3,1,1.0,0.833,0.231
179,Scott Machado,4,19.166667,6,1,1.0,0.776,0.130
321,Damian Jones,24,409.618333,74,0,0.0,0.720,0.113


,PLAYER_NAME,GP,MIN,FGA,FG3A,FG3_PCT,TS_PCT,USG_PCT,AST_PCT,POSS
459,Shai Gilgeous-Alexander,82,2173.906667,716,139,0.367,0.554,0.182,0.179,4669


### Step 1B findings

- **One row per player-season**: traded players are already consolidated (`TEAM_COUNT` > 1), and `TEAM_ABBREVIATION` is their *last* team. That's fine for player features, wrong for team context.
- **`_PCT` columns are fractions** (USG 0.182, not 18.2).
- **Undefined percentages come back as `0.0`, not NaN**: Zhou Qi and Damian Jones show `FG3_PCT = 0.0` on 0 attempts. When cleaning, we'll recompute every percentage from makes and attempts, so 0/0 becomes NaN.
- **Small samples dominate the top of every rate stat** (e.g. TS% 1.500 on 1 FGA). Plan: empirical-Bayes shrinkage, with each stat shrunk according to its own denominator (FGA for TS%, possessions for USG%), rather than a hard minutes cutoff.
- **Missing from this endpoint:** height, weight, position, birth date, draft info. Those come in Step 1C.